# Batch Feature Extraction With Train/Test Splits

This notebook prepares the feature table for modelling. It creates subject-level train/test and cross-validation fold assignments, then runs feature extraction in a resumable way.

The important leakage rule is that the same participant must never appear in both training and test data. Since each participant can contribute both eyes-open and eyes-closed rows, splitting is done at the subject level first, then applied back to the epoch files.


## 1. Setup

This notebook keeps configuration visible and uses helper functions from `src/eeg_feature_extraction.py` for the repeated feature extraction work.


In [ ]:
from importlib import reload
from pathlib import Path
import os
import sys
import time

import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.eeg_feature_extraction as efe
from src import metadata_utils

reload(efe)
reload(metadata_utils)

processed_eeg_dir = project_root / "data" / "processed" / "eeg"
metadata_path = project_root / "data" / "participant_metadata" / "participants.tsv"
feature_dir = project_root / "data" / "features"
feature_dir.mkdir(parents=True, exist_ok=True)

split_path = feature_dir / "rest_eeg_subject_splits.tsv"
train_features_path = feature_dir / "rest_eeg_features_train.tsv"
test_features_path = feature_dir / "rest_eeg_features_test.tsv"
all_features_path = feature_dir / "rest_eeg_features_split.tsv"
microstate_dir = feature_dir / "microstates"
microstate_dir.mkdir(parents=True, exist_ok=True)
microstate_features_path = microstate_dir / "rest_eeg_microstate_features.tsv"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_CV_FOLDS = 3
SPLIT_STRATIFY_COL = "APOE_group"
RUN_ALL_FEATURES = False
RUN_MICROSTATES = False

print(f"Processed EEG dir: {processed_eeg_dir}")
print(f"Feature dir: {feature_dir}")
print(f"Microstate dir: {microstate_dir}")


## 2. Feature Settings

These settings match the feature extraction notebook. They are kept here because they define the actual feature space used for modelling.


In [ ]:
EEG_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma_low": (30.0, 40.0),
}

CONNECTIVITY_BANDS = {
    "theta": EEG_BANDS["theta"],
    "alpha": EEG_BANDS["alpha"],
    "beta": EEG_BANDS["beta"],
    "gamma_low": EEG_BANDS["gamma_low"],
}

REGION_PREFIXES = {
    "frontal": ("Fp", "AF", "AFF", "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "F9", "F10", "Fz", "FFC", "FFT"),
    "central": ("FC", "FCC", "C1", "C2", "C3", "C4", "C5", "C6", "Cz", "CCP"),
    "temporal": ("FT", "FTT", "T7", "T8", "TTP", "TP", "TPP"),
    "parietal": ("CP", "CPP", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "Pz", "PPO"),
    "occipital": ("PO", "POO", "O", "OI"),
}

epoch_index_preview = efe.find_clean_epoch_files(processed_eeg_dir)
if epoch_index_preview.empty:
    raise RuntimeError("No cleaned epoch files found. Finish preprocessing before running feature extraction.")

example_epochs = efe.load_epochs(epoch_index_preview.iloc[0]["path"])
all_channels = sorted({
    ch
    for path in epoch_index_preview["path"]
    for ch in efe.load_epochs(path).copy().pick("eeg", exclude=[]).ch_names
})

REGIONS = efe.make_regions_from_prefixes(all_channels, REGION_PREFIXES)

efe.EEG_BANDS = EEG_BANDS
efe.CONNECTIVITY_BANDS = CONNECTIVITY_BANDS
efe.REGIONS = REGIONS

display(efe.feature_family_table())
display(efe.region_channel_table(example_epochs))


## 3. Find Clean Epoch Files

Each row is one cleaned subject-condition file. Feature extraction produces one feature row per subject and condition.


In [ ]:
epoch_index = efe.find_clean_epoch_files(processed_eeg_dir)
print(f"Found {len(epoch_index)} cleaned epoch files across {epoch_index['subject_id'].nunique()} subjects.")
display(epoch_index.head())

condition_counts = epoch_index.groupby("subject_id")["condition"].nunique().value_counts().sort_index()
display(condition_counts.to_frame("n_subjects"))


## 4. Create Subject-Level Train/Test And CV Splits

The split is created at the participant level to avoid leakage between eyes-open and eyes-closed rows from the same person. The held-out test set should stay untouched until final model evaluation.

Three CV folds are used by default inside the training set. Five-fold CV is not usually too expensive once features are already extracted, but three folds is a pragmatic starting point while the pipeline is still being developed.


In [ ]:
full_cohort, participants = metadata_utils.prepare_imaging_metadata(metadata_path)
participants = participants.loc[participants["participant_id"].isin(epoch_index["subject_id"])].copy()

split_subjects = (
    epoch_index[["subject_id"]]
    .drop_duplicates()
    .merge(participants, left_on="subject_id", right_on="participant_id", how="left")
)

if split_subjects[SPLIT_STRATIFY_COL].isna().any():
    missing = split_subjects.loc[split_subjects[SPLIT_STRATIFY_COL].isna(), "subject_id"].tolist()
    raise RuntimeError(f"Missing stratification labels for: {missing}")

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

subject_array = split_subjects["subject_id"].to_numpy()
y = split_subjects[SPLIT_STRATIFY_COL].to_numpy()
train_idx, test_idx = next(splitter.split(subject_array, y))

split_subjects["split"] = ""
split_subjects.loc[train_idx, "split"] = "train"
split_subjects.loc[test_idx, "split"] = "test"
split_subjects["cv_fold"] = pd.NA

train_subjects = split_subjects.loc[split_subjects["split"] == "train"].copy()
min_class_count = train_subjects[SPLIT_STRATIFY_COL].value_counts().min()
n_folds = min(N_CV_FOLDS, int(min_class_count))
if n_folds < 2:
    raise RuntimeError("Not enough subjects per class to create cross-validation folds.")

cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
for fold, (_, val_idx) in enumerate(cv.split(train_subjects["subject_id"], train_subjects[SPLIT_STRATIFY_COL]), start=1):
    val_subjects = train_subjects.iloc[val_idx]["subject_id"]
    split_subjects.loc[split_subjects["subject_id"].isin(val_subjects), "cv_fold"] = fold

split_subjects.to_csv(split_path, sep="	", index=False)

print(f"Saved subject splits to: {split_path}")
display(pd.crosstab(split_subjects["split"], split_subjects[SPLIT_STRATIFY_COL]))
display(pd.crosstab(split_subjects["cv_fold"], split_subjects[SPLIT_STRATIFY_COL], dropna=False))


## 5. Apply Splits To Epoch Files

This table shows which cleaned epoch files belong to the training set and which belong to the held-out test set.


In [ ]:
epoch_index_split = epoch_index.merge(
    split_subjects[["subject_id", "split", "cv_fold", SPLIT_STRATIFY_COL]],
    on="subject_id",
    how="inner",
)

print(epoch_index_split["split"].value_counts().to_string())
display(epoch_index_split.head())

train_subject_ids = set(split_subjects.loc[split_subjects["split"] == "train", "subject_id"])
test_subject_ids = set(split_subjects.loc[split_subjects["split"] == "test", "subject_id"])


## 6. Time One Feature Row

This estimates how long a full run may take on this laptop. Connectivity and graph metrics are usually the slowest part.


In [ ]:
test_row = epoch_index_split.iloc[0]
start = time.time()
_ = efe.extract_epoch_file_features(test_row)
elapsed = time.time() - start
estimated_minutes = elapsed * len(epoch_index_split) / 60

print(f"One subject-condition file took {elapsed:.1f} seconds")
print(f"Estimated full extraction time: {estimated_minutes:.1f} minutes for {len(epoch_index_split)} files")


## 7. Extract Training Features First

This creates the training feature table first. Progress is saved after every subject-condition row, so rerunning the cell with `RUN_ALL_FEATURES = False` resumes rather than starting again.


In [ ]:
train_features = efe.extract_features_resumable(
    epoch_index_split,
    train_features_path,
    subject_ids=train_subject_ids,
    run_all=RUN_ALL_FEATURES,
)

print(train_features.shape)
display(train_features.head())


## 8. Extract Held-Out Test Features

These features are extracted with the same fixed settings. Do not use the held-out test labels for model selection or feature selection later.


In [ ]:
test_features = efe.extract_features_resumable(
    epoch_index_split,
    test_features_path,
    subject_ids=test_subject_ids,
    run_all=RUN_ALL_FEATURES,
)

print(test_features.shape)
display(test_features.head())


## 9. Add Training-Derived Microstate Features

Microstate features are handled after the subject-level split because the templates are learned from data. To avoid leakage into the final held-out test set, the templates are fitted using training subjects only. The fitted templates are then backfitted to both training and test subjects so the resulting microstate features are comparable.

This step is optional and off by default because it is more computationally intensive than the faster feature families. Set `RUN_MICROSTATES = True` when you want to run it.


In [ ]:
if RUN_MICROSTATES:
    # Fit templates once using training subjects only.
    microstate_model, microstate_fit_summary = efe.fit_microstate_templates(
        epoch_index_split,
        train_subject_ids,
        microstate_dir,
        condition="eyes_closed",
        n_clusters=4,
        n_init=10,
        random_state=RANDOM_STATE,
    )

    # Backfit the training-derived templates to all subjects.
    microstate_features = efe.backfit_microstate_features(
        epoch_index_split,
        microstate_model,
        microstate_features_path,
        subject_ids=train_subject_ids | test_subject_ids,
        condition="eyes_closed",
        run_all=RUN_ALL_FEATURES,
    )

    display(microstate_fit_summary.to_frame("value"))
    display(microstate_features.head())
else:
    print("Microstate extraction skipped. Set RUN_MICROSTATES = True to fit training-derived templates and backfit features.")


## 10. Combine Features With Split Labels

The combined table is convenient for later modelling notebooks, but model code should still respect the `split`, `cv_fold`, and `subject_id` columns.


In [ ]:
features = pd.concat([train_features, test_features], ignore_index=True)

if microstate_features_path.exists():
    microstate_features = pd.read_csv(microstate_features_path, sep="\t")
    features = features.merge(
        microstate_features,
        on=["subject_id", "condition"],
        how="left",
        suffixes=("", "_microstate"),
    )

features = features.merge(
    split_subjects[["subject_id", "split", "cv_fold", "APOE_group", "PICALM_group", "APOE_risk_dosage", "PICALM_risk_dosage", "age", "sex", "education"]],
    on="subject_id",
    how="left",
)
features = features.sort_values(["split", "subject_id", "condition"]).reset_index(drop=True)
features.to_csv(all_features_path, sep="\t", index=False)

print(f"Saved combined split feature table to: {all_features_path}")
print(features.shape)
display(features[["subject_id", "condition", "split", "cv_fold", "APOE_group", "PICALM_group"]].head(20))


## 11. Checks Before Modelling

These checks make sure the split is subject-clean and that no participant appears in both train and test.


In [ ]:
train_ids = set(features.loc[features["split"] == "train", "subject_id"])
test_ids = set(features.loc[features["split"] == "test", "subject_id"])
overlap = train_ids & test_ids

print(f"Train subjects: {len(train_ids)}")
print(f"Test subjects: {len(test_ids)}")
print(f"Train/test overlap: {len(overlap)}")

if overlap:
    raise RuntimeError(f"Subject leakage detected: {sorted(overlap)}")

missing_by_col = features.isna().mean().sort_values(ascending=False)
display(missing_by_col[missing_by_col > 0].to_frame("missing_fraction").head(30))


## Notes For The Modelling Notebook

For the next modelling notebook, fit scalers, imputers, feature selection, and models on the training folds only. The held-out test set should be used once at the end for final evaluation.

Most current features are computed independently per subject-condition file, so they do not require train-only fitting. Microstates are the exception because the templates are learned from data. This notebook handles the final hold-out test set safely by fitting templates on training subjects only and backfitting them to held-out subjects.

For fully leakage-safe cross-validation estimates involving microstate features, the strictest approach is to fit separate microstate templates inside each training fold and backfit them to that fold's validation subjects. That is more computationally expensive, so this notebook keeps the first implementation focused on the final train/test workflow.
